# 🎙️ AUDIO AI AGENT (FULL QWEN HYBRID) 🤖
Notebook ini mengkonversi file audio menjadi transkrip menggunakan Whisper/Qwen ASR, lalu membuat notulen rapat berformat JSON menggunakan AI Agent (Ollama/Qwen LLM).

In [13]:
# Install dependencies
!pip install openai-whisper openai

You should consider upgrading via the '/Users/isasubani/Documents/MyProductivityApp/venv/bin/python3 -m pip install --upgrade pip' command.


# Config

In [ ]:
import os
import json
import whisper
from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    if os.path.exists(".env"):
        with open(".env", "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, val = line.split("=", 1)
                    os.environ[key.strip()] = val.strip().strip('"').strip("'")

# =======================================================
# 🎛️ PANEL KENDALI HYBRID (CLOUD vs LOCAL)
# =======================================================
TRANSCRIBER_MODE = "local"  
AI_AGENT_MODE = "cloud"     

# --- KREDENSIAL ALIBABA DASHSCOPE (SINGAPURA) ---
ALIBABA_API_KEY = os.getenv("CLOUD_LLM_KEY")
ALIBABA_BASE_URL = os.getenv("CLOUD_LLM_URL")

# 1. KONFIGURASI TRANSCRIBER (AUDIO TO TEXT)
LOCAL_WHISPER_MODEL = os.getenv("LOCAL_STT_MODEL", "base") 
CLOUD_STT_API_KEY = os.getenv("CLOUD_STT_KEY", ALIBABA_API_KEY)
CLOUD_STT_BASE_URL = os.getenv("CLOUD_STT_URL")
CLOUD_STT_MODEL = os.getenv("CLOUD_STT_MODEL") 

# 2. KONFIGURASI AI AGENT (TEKS TO JSON NOTULEN)
OLLAMA_BASE_URL = os.getenv("LOCAL_LLM_URL", "http://localhost:11434/v1")
OLLAMA_MODEL = os.getenv("LOCAL_LLM_MODEL", "llama3") 
OLLAMA_API_KEY = os.getenv("LOCAL_LLM_KEY", "ollama") 
CLOUD_AI_API_KEY = os.getenv("CLOUD_LLM_KEY", ALIBABA_API_KEY)
CLOUD_AI_BASE_URL = os.getenv("CLOUD_LLM_URL", ALIBABA_BASE_URL)
CLOUD_AI_MODEL = os.getenv("CLOUD_LLM_MODEL", "qwen3.7-plus")

# Prepare Class

In [15]:
class AudioTranscriber:
    def __init__(self, mode, local_model, cloud_key, cloud_url, cloud_model):
        self.mode = mode
        self.local_model = local_model
        self.cloud_model = cloud_model
        
        self.local_whisper_client = None
        self.cloud_client = None
        
        if self.mode == "cloud":
            if not cloud_key:
                raise ValueError("API Key Cloud STT kosong!")
            self.cloud_client = OpenAI(api_key=cloud_key, base_url=cloud_url)

    def transcribe(self, audio_path):
        """Memproses audio menjadi teks secara Hybrid."""
        print(f"[⏳] Sedang memproses transkripsi...")
        
        if self.mode == "local":
            print(f"⚙️ Mode: LOKAL (Whisper Mac M2 - {self.local_model})")
            if self.local_whisper_client is None:
                self.local_whisper_client = whisper.load_model(self.local_model)
            
            result = self.local_whisper_client.transcribe(audio_path, language="id")
            return result["text"]
            
        elif self.mode == "cloud":
            print(f"⚙️ Mode: CLOUD (Qwen ASR - {self.cloud_model})")
            with open(audio_path, "rb") as audio_file:
                transcription = self.cloud_client.audio.transcriptions.create(
                    model=self.cloud_model,
                    file=audio_file,
                    language="id"
                )
            return transcription.text

In [16]:
class AIAgent:
    def __init__(self, mode, ollama_url, ollama_model, ollama_key, cloud_url, cloud_model, cloud_key):
        self.mode = mode
        
        if self.mode == "local":
            print(f"🤖 Inisialisasi AI Agent LOKAL (Ollama - {ollama_model})...")
            self.client = OpenAI(api_key=ollama_key, base_url=ollama_url)
            self.model_name = ollama_model
        elif self.mode == "cloud":
            print(f"☁️ Inisialisasi AI Agent CLOUD (Qwen - {cloud_model})...")
            self.client = OpenAI(api_key=cloud_key, base_url=cloud_url)
            self.model_name = cloud_model
        else:
            raise ValueError("Mode AI Agent tidak valid!")

    def extract_to_json(self, text):
        """Merangkum transkrip meeting ke format JSON."""
        print(f"[⏳] Mengirim data ke AI Agent...")
        
        system_prompt = """Kamu adalah asisten AI cerdas yang bertugas sebagai notulen rapat. 
Tugasmu adalah menganalisis transkrip rapat dan menghasilkan ringkasan dalam format JSON yang valid.

ATURAN MUTLAK:
1. Output HANYA boleh berupa JSON, tanpa awalan atau akhiran teks apapun.
2. Jika ada informasi yang tidak dibahas (misal tidak ada action items), biarkan array-nya kosong [].

STRUKTUR JSON YANG DIWAJIBKAN:
{
  "ringkasan_umum": "Deskripsi singkat 2-3 kalimat tentang inti dari rapat tersebut.",
  "poin_penting": [
    "Poin pembahasan utama 1",
    "Poin pembahasan utama 2"
  ],
  "action_items": [
    "Tugas 1 - [Nama Orang jika disebutkan]",
    "Tugas 2 - [Nama Orang jika disebutkan]"
  ]
}"""

        user_prompt = f"Tolong buatkan ringkasan dari transkrip rapat berikut ke dalam format JSON:\n\n{text}"

        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt}
                ],
                response_format={ "type": "json_object" } 
            )
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            return f"[ERROR] API ({self.mode}) Gagal: {e}"

# Input

In [17]:
AUDIO_FILE = "/Users/isasubani/Documents/test transcript/Daily 22 Juni 2026.m4a" 
TRANSCRIPT_FILE = "/Users/isasubani/Documents/test transcript/Transkrip_Daily_22_Juni_2026.txt"

# Transcrip

In [18]:
import os

if not os.path.exists(AUDIO_FILE):
    print(f"❌ File tidak ditemukan di {AUDIO_FILE}. Pastikan path sudah benar ya, bro!")
else:
    # 1. Jalankan Transcriber
    transcriber = AudioTranscriber(
        mode=TRANSCRIBER_MODE,
        local_model=LOCAL_WHISPER_MODEL,
        cloud_key=CLOUD_STT_API_KEY,
        cloud_url=CLOUD_STT_BASE_URL,
        cloud_model=CLOUD_STT_MODEL
    )
    
    hasil_teks = transcriber.transcribe(AUDIO_FILE)
    
    # 2. Simpan hasil teks ke file
    with open(TRANSCRIPT_FILE, "w", encoding="utf-8") as f:
        f.write(hasil_teks)
        
    print(f"\n✅ BERHASIL! Transkrip disimpan di file: {TRANSCRIPT_FILE}")
    print("-" * 40)
    print(hasil_teks[:500] + "...\n(Teks dipotong untuk tampilan)")

[⏳] Sedang memproses transkripsi...
⚙️ Mode: LOKAL (Whisper Mac M2 - base)


/Users/isasubani/Documents/MyProductivityApp/venv/lib/python3.9/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



✅ BERHASIL! Transkrip disimpan di file: /Users/isasubani/Documents/test transcript/Transkrip_Daily_22_Juni_2026.txt
----------------------------------------
 jadi dongbul kayak gitu oke ini juga kan ini juga konfig mas 30 menit konfig? konfig konfig konfig jadi nanti kalau misalkan dia udah besok itu 30 menit ya nanti bakal request lagi dia kembali ke permutansi pertama itu sih masih itu di fitma udah lehl sama di mobil kok permutansinya kayak gitu visualnya ya gitu oke oke oke batik secara requirementnya kemarin itu aku butuhnya di satu menit mungkin nanti itu di aku perfaiknya di 30 menit batiknya masih untuk kota ipinnya oke itu untuk request sam...
(Teks dipotong untuk tampilan)


# Discussion with AI

In [19]:
import json
import os

# Pastikan nama file sama dengan output di Cell 6A

if not os.path.exists(TRANSCRIPT_FILE):
    print(f"❌ File teks tidak ditemukan di {TRANSCRIPT_FILE}. Pastikan jalankan Cell 6A dulu, bro!")
else:
    # 1. Baca teks dari file
    with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
        teks_transkrip = f.read()

    print(f"\n[⏳] Membaca file teks ({len(teks_transkrip)} karakter) dan memproses Notulen AI...")
    print("=" * 40)
    
    # 2. Jalankan AI Agent Extract to JSON
    agent = AIAgent(
        mode=AI_AGENT_MODE,
        ollama_url=OLLAMA_BASE_URL, ollama_model=OLLAMA_MODEL, ollama_key=OLLAMA_API_KEY,
        cloud_url=CLOUD_AI_BASE_URL, cloud_model=CLOUD_AI_MODEL, cloud_key=CLOUD_AI_API_KEY
    )
    
    hasil_json = agent.extract_to_json(teks_transkrip)
    
    print("\n📊 HASIL NOTULEN JSON:")
    print("-" * 40)
    
    # 3. Parsing dan print JSON supaya formatnya rapi
    try:
        parsed_json = json.loads(hasil_json)
        print(json.dumps(parsed_json, indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print("❌ AI gagal me-return format JSON yang valid. Raw text:")
        print(hasil_json)


[⏳] Membaca file teks (16615 karakter) dan memproses Notulen AI...
☁️ Inisialisasi AI Agent CLOUD (Qwen - qwen3.7-plus)...
[⏳] Mengirim data ke AI Agent...

📊 HASIL NOTULEN JSON:
----------------------------------------
{
  "ringkasan_umum": "Rapat membahas konfigurasi keamanan OTP, mekanisme deep linking untuk pelacakan promo dan event dari pihak ketiga seperti Insider, serta peninjauan UI/UX dan integrasi data untuk halaman pengajuan produk. Tim juga menyepakati perbaikan rute navigasi banner dan penambahan fitur riwayat pengajuan serta simulasi cicilan pada aplikasi.",
  "poin_penting": [
    "Konfigurasi batas percobaan OTP (3 kali salah) dan durasi blokir (3 atau 30 menit) untuk fitur register, reset PIN, dan ganti nomor HP.",
    "Implementasi deep linking dan tracking ID untuk promo dan event dari sumber eksternal (seperti Insider) agar langsung mengarah ke halaman detail yang spesifik.",
    "Penyesuaian rute navigasi banner agar dapat langsung membuka halaman pengajuan produk